# Модель на CoBaLD

In [ ]:
pip install pytorch-crf

In [ ]:
pip install pyconll

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd
import pyconll

from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
from transformers import BertModel

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from collections import Counter
import numpy as np
import random
import os

# Устанавливаем сиды для воспроизводимости
def set_seed(seed_value=12345):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True

set_seed(12345)

class CustomCoNLLDataset(Dataset):
    def __init__(self, conllu_file, tokenizer, max_length=256, target_column=-1):
        self.data, self.labels = [], set()
        current_sentence, current_labels = [], []
        with open(conllu_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    if not line and current_sentence:
                        self.data.append((current_sentence.copy(), current_labels.copy()))
                        current_sentence, current_labels = [], []
                    continue
                parts = line.split('\t')
                if parts[0].isdigit() or '-' in parts[0]:
                    word = parts[1]
                    sem_class = parts[target_column] if len(parts) > 10 else 'O'
                    current_sentence.append(word)
                    current_labels.append(sem_class)
                    self.labels.add(sem_class)
            if current_sentence:
                self.data.append((current_sentence, current_labels))

        # Сортируем метки для консистентного маппинга
        self.tokenizer = tokenizer
        self.max_length = max_length
        # Анализируем распределение меток
        flat_labels = [lbl for _, labels in self.data for lbl in labels]
        self.label_counts = Counter(flat_labels)
        print(f"Label distribution: {self.label_counts}")

        # O должен быть на первой позиции, если он есть
        sorted_labels = sorted(self.labels)
        if 'O' in sorted_labels:
            sorted_labels.remove('O')
            sorted_labels = ['O'] + sorted_labels

        self.label2id = {l: i for i, l in enumerate(sorted_labels)}
        self.id2label = {i: l for l, i in self.label2id.items()}
        print(f"Labels: {self.label2id}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, labels = self.data[idx]
        text = ' '.join(tokens)

        # Более надежный контроль токенизации
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt',
            return_offsets_mapping=True,
            return_special_tokens_mask=True,
            is_split_into_words=False
        )

        # Подготовка меток
        token_labels = torch.ones(self.max_length, dtype=torch.long) * -100

        # Пословная токенизация
        token_to_word_mapping = {}
        word_idx = 0
        for i, token in enumerate(self.tokenizer.convert_ids_to_tokens(encoding['input_ids'][0])):
            # Пропускаем специальные токены
            if encoding['special_tokens_mask'][0][i] == 1:
                continue

            # Определяем, к какому слову относится токен
            if token.startswith('##'):
                if i > 0 and i-1 in token_to_word_mapping:
                    token_to_word_mapping[i] = token_to_word_mapping[i-1]
            else:
                # Новое слово
                if word_idx < len(labels):
                    token_to_word_mapping[i] = word_idx
                    word_idx += 1

        # Присваиваем метки
        for token_idx, word_idx in token_to_word_mapping.items():
            if word_idx < len(labels):
                token_labels[token_idx] = self.label2id.get(labels[word_idx], 0)

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': token_labels
        }


class SemanticModel(nn.Module):
    def __init__(self, model_name, num_labels, dropout_rate=0.2):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

        # Инициализируем веса классификатора для лучшей сходимости
        self.classifier.weight.data.normal_(mean=0.0, std=0.02)
        self.classifier.bias.data.zero_()

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(logits.view(-1, logits.shape[-1]), labels.view(-1))

        return {'loss': loss, 'logits': logits} if loss is not None else {'logits': logits}


def train_and_eval_semantic(conllu_train,
                           conllu_dev,
                           model_name='DeepPavlov/rubert-base-cased',
                           device=None,
                           epochs=15,
                           batch_size=8,
                           max_length=256,
                           lr=5e-5,
                           warmup_ratio=0.1,
                           weight_decay=0.01,
                           gradient_accumulation_steps=2,
                           early_stopping_patience=3):

    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Токенайзер и датасеты
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    print("Loading datasets...")
    train_ds = CustomCoNLLDataset(conllu_train, tokenizer, max_length=max_length)
    dev_ds = CustomCoNLLDataset(conllu_dev, tokenizer, max_length=max_length)

    # Проверяем, что лейблы совпадают
    if train_ds.label2id != dev_ds.label2id:
        print("WARNING: Label mappings differ between train and dev sets!")
        print(f"Train: {train_ds.label2id}")
        print(f"Dev: {dev_ds.label2id}")
        # Используем маппинг из тренировочного набора для обоих
        dev_ds.label2id = train_ds.label2id
        dev_ds.id2label = train_ds.id2label

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    dev_dl = DataLoader(dev_ds, batch_size=batch_size*2, shuffle=False)

    # Вычисляем веса классов для борьбы с дисбалансом
    num_classes = len(train_ds.label2id)
    label_counts = Counter()
    for _, labels in train_ds.data:
        label_counts.update(labels)

    total = sum(label_counts.values())
    # Веса обратно пропорциональны частоте класса
    class_weights = torch.ones(num_classes, device=device)
    for label, idx in train_ds.label2id.items():
        if label in label_counts and label_counts[label] > 0:
            class_weights[idx] = total / (num_classes * label_counts[label])

    # Ограничиваем максимальный вес, чтобы избежать числовой нестабильности
    class_weights = torch.clamp(class_weights, 0.1, 10.0)
    print(f"Class weights: {class_weights}")

    # Создаем модель с нуля
    model = SemanticModel(model_name, num_classes, dropout_rate=0.3).to(device)

    # Настраиваем оптимизатор с различными скоростями обучения
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.bert.named_parameters()
                      if not any(nd in n for nd in no_decay)],
            'weight_decay': weight_decay,
            'lr': lr
        },
        {
            'params': [p for n, p in model.bert.named_parameters()
                      if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0,
            'lr': lr
        },
        {
            'params': [p for n, p in model.classifier.named_parameters()],
            'weight_decay': weight_decay,
            'lr': lr * 10  # Более высокая скорость для классификатора
        }
    ]

    optimizer = AdamW(optimizer_grouped_parameters)

    # Настраиваем расписание обучения
    total_steps = epochs * len(train_dl) // gradient_accumulation_steps
    warmup_steps = int(warmup_ratio * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # Обучение и валидация
    best_f1 = 0.0
    no_improvement_count = 0

    for epoch in range(1, epochs+1):
        model.train()
        train_loss = 0.0
        optimizer.zero_grad()

        for step, batch in enumerate(train_dl):
            # Перемещаем данные на устройство
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Прямой проход
            outputs = model(input_ids, attention_mask, labels)
            loss = outputs['loss'] / gradient_accumulation_steps
            loss.backward()

            train_loss += loss.item() * gradient_accumulation_steps

            # Обновление весов каждые gradient_accumulation_steps шагов
            if (step + 1) % gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        train_loss = train_loss / len(train_dl)
        print(f"[Train] Epoch {epoch}/{epochs}  Loss={train_loss:.4f}")

        model.eval()
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in dev_dl:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].cpu().numpy()

                outputs = model(input_ids, attention_mask)
                logits = outputs['logits']
                preds = torch.argmax(logits, dim=-1).cpu().numpy()

                for i in range(preds.shape[0]):
                    pred = preds[i][batch['attention_mask'][i] == 1]
                    label = labels[i][batch['attention_mask'][i] == 1]

                    # Фильтруем специальные токены
                    mask = label != -100
                    all_preds.extend(pred[mask])
                    all_labels.extend(label[mask])

        # Рассчитываем метрики
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro')

        # Показываем отчет по классам для более глубокого анализа
        if epoch % 5 == 0 or epoch == epochs:
            # Получаем только присутствующие в данных классы
            unique_labels = sorted(set(all_labels + all_preds))
            actual_label_names = [train_ds.id2label[i] for i in unique_labels if i in train_ds.id2label]

            report = classification_report(
                all_labels, all_preds,
                labels=unique_labels,  # Используем только реальные метки
                target_names=actual_label_names,  # Используем только реальные имена
                digits=4
            )
            print(f"Classification Report:\n{report}")

        print(f"[Dev]   Epoch {epoch}/{epochs}  Acc={acc:.4f}  F1={f1:.4f}")

        # Сохраняем лучшую модель
        if f1 > best_f1:
            best_f1 = f1
            torch.save({
                'model_state_dict': model.state_dict(),
                'label2id': train_ds.label2id,
                'id2label': train_ds.id2label,
                'f1': f1,
                'acc': acc
            }, 'best_semantic_model.pt')
            print(f"New best model saved (F1={best_f1:.4f})")
            no_improvement_count = 0
        else:
            no_improvement_count += 1

        # Ранняя остановка
        if no_improvement_count >= early_stopping_patience:
            print(f"Early stopping after {epoch} epochs without improvement")
            break

    # Загружаем лучшую модель
    checkpoint = torch.load('best_semantic_model.pt')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    print(f"Best model F1: {checkpoint['f1']:.4f}, Acc: {checkpoint['acc']:.4f}")
    return model, tokenizer, checkpoint['label2id']


class SarcasmDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df['text'].tolist()
        self.labels = df['sarcasm'].tolist()
        self.sem_feats = df['sem_pooled'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'sem_feats': torch.tensor(self.sem_feats[idx], dtype=torch.float),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }


class SarcasmClassifier(nn.Module):
    def __init__(self, bert_model, sem_dim, num_classes):
        super().__init__()
        self.bert = bert_model
        self.sem_proj = nn.Linear(sem_dim, bert_model.config.hidden_size)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(bert_model.config.hidden_size*2, num_classes)

    def forward(self, input_ids, attention_mask, sem_feats, labels=None):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask).pooler_output
        sem_proj = self.sem_proj(sem_feats)
        joint = torch.cat([bert_out, sem_proj], dim=1)
        joint = self.dropout(joint)
        logits = self.classifier(joint)

        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            return {'loss': loss, 'logits': logits}
        return {'logits': logits}


# Пример использования:
if __name__ == "__main__":
    # Устройство
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Пути к данным
    train_conllu = r'/content/train.conllu'
    dev_conllu = r'/content/dev.conllu'

    # Обучение семантической модели с улучшенными параметрами
    sem_model, tokenizer, label2id = train_and_eval_semantic(
        conllu_train=train_conllu,
        conllu_dev=dev_conllu,
        model_name='DeepPavlov/rubert-base-cased',
        device=device,
        epochs=5,
        batch_size=8,
        max_length=256,
        lr=5e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        gradient_accumulation_steps=2,
        early_stopping_patience=3
    )

    # Модель готова к использованию
    sem_model.eval()

Using device: cuda
Loading datasets...
Label distribution: Counter({'_': 80540, 'PREPOSITION': 45482, 'BEING': 45479, 'CH_REFERENCE_AND_QUANTIFICATION': 30680, 'ORGANIZATION': 17512, 'TIME': 12358, 'COUNTRY_AS_ADMINISTRATIVE_UNIT': 10350, 'VERBAL_COMMUNICATION': 9703, 'COORDINATING_CONJUNCTIONS': 8507, 'CONJUNCTIONS': 5416, 'CH_OF_CONNECTIONS': 5414, 'DISCOURSIVE_UNITS': 5387, 'MODALITY': 5344, 'INHABITED_LOCALITY': 5091, 'PARTICLES': 4907, 'ENTITY_OR_SITUATION_PRONOUN': 4155, 'MOTION': 3883, 'AUXILIARY_VERBS': 3344, 'CH_DEGREE': 3221, 'ARRANGEMENTS': 3155, 'BE': 3081, 'MONEY': 2972, 'TO_COMMIT': 2769, 'RESULTS_OF_GIVING_INFORMATION_AND_SPEECH_ACTIVITY': 2716, 'TRANSPORT': 2682, 'TO_GIVE': 2646, 'STATE_OF_MIND': 2553, 'TO_TAKE_PLACE': 2321, 'CH_DISPOSITION_AND_MOTION': 2274, 'PHYSICAL_PSYCHIC_CONDITION': 2169, 'POSITION_IN_SPACE': 1938, 'DOCUMENT': 1878, 'INFORMATION': 1829, 'EMOTIONS_AND_THEIR_EXPRESSION': 1818, 'PLACE': 1795, 'CIRCUMSTANCE': 1783, 'LAWS_AND_STANDARDS': 1710, 'EXISTEN

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[Train] Epoch 1/3  Loss=1.2077


In [ ]:
# Загружаем сырые данные
df = pd.read_csv('/content/dataset_all_data (2).csv')
df['sarcasm'] = df['sarcasm'].astype(int)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Пред–вычисление sem_pooled
sem_model.eval().to(device)
sem_pooled_list = []
for text in df['text'].tolist():
    enc = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        logits = sem_model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask']
        )['logits']           # [1, seq_len, sem_dim]
    pooled = logits.squeeze(0).mean(dim=0).cpu().numpy()  # [sem_dim]
    sem_pooled_list.append(pooled)

df['sem_pooled'] = sem_pooled_list

# DataLoader
ds = SarcasmDataset(df, tokenizer, max_length=128)
dl = DataLoader(ds, batch_size=16, shuffle=True)

# Создаём и обучаем SarcasmClassifier
bert = sem_model.bert
sarcasm_model = SarcasmClassifier(
    bert_model=bert,
    sem_dim=df['sem_pooled'].iloc[0].shape[0],
    num_classes=2
).to(device)

for param in sarcasm_model.bert.parameters():
    param.requires_grad = False

#optimizer = AdamW(sarcasm_model.parameters(), lr=2e-5)

optimizer = AdamW(
    filter(lambda p: p.requires_grad, sarcasm_model.parameters()),
    lr=2e-5
)

sarcasm_model.train()
for epoch in range(3):
    total_loss = 0.0
    for batch in dl:
        optimizer.zero_grad()
        out = sarcasm_model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            sem_feats=batch['sem_feats'].to(device),
            labels=batch['labels'].to(device)
        )
        out['loss'].backward()
        optimizer.step()
        total_loss += out['loss'].item()
    print(f"Sarcasm Epoch {epoch+1}, Loss={total_loss/len(dl):.4f}")

# Сохраняем модель
torch.save(sarcasm_model.state_dict(), 'sarcasm_model.pt')
print("Training complete.")

In [ ]:
# Генерация sem_pooled для всех текстов — выполняем один раз
sem_pooled_list = []
sem_model.eval()
for text in df['text'].tolist():
    enc = tokenizer(
        text,
        truncation=True,
        padding='max_length',
        max_length=128,
        return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        logits = sem_model(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask']
        )['logits']  # [1, seq_len, num_labels]

    # Усредняем по длине последовательности
    pooled = logits.squeeze(0).mean(dim=0).cpu().numpy()  # [num_labels]
    sem_pooled_list.append(pooled)

# Сохраняем в df и на диск
df['sem_pooled'] = sem_pooled_list
df.to_pickle('enriched_sarcasm_pooled.pkl')
print("Pre-computed semantic features for", len(df), "texts.")

In [ ]:
abc = pd.read_pickle(r'/content/enriched_sarcasm_pooled.pkl')

In [ ]:
abc

## Обучение

In [ ]:
import random

In [ ]:
def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_random_seed(12345)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro')
    }

# Загружаем обогащённый датасет
df = pd.read_pickle('enriched_sarcasm_pooled.pkl')
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['sarcasm'], random_state=42)

tokenizer = BertTokenizer.from_pretrained('DeepPavlov/rubert-base-cased')
train_ds = SarcasmDataset(train_df, tokenizer)
eval_ds  = SarcasmDataset(test_df,  tokenizer)

bert = BertModel.from_pretrained('DeepPavlov/rubert-base-cased')

model = SarcasmClassifier(
    bert_model=bert,
    sem_dim=len(df['sem_pooled'].iloc[0]),
    num_classes=2
)

training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2,

    # частота в шагах
    logging_steps=50,
    eval_steps=500,
    save_steps=500,
    save_total_limit=1,

    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics
)

# Тренировка и оценка
trainer.train()
metrics = trainer.evaluate()
print(metrics)

In [ ]:
'''import torch
import torch.nn as nn
from transformers import BertModel, BertConfig

class CustomBertClassifier(nn.Module):
    def __init__(self,
                 pretrained_model_name: str = 'bert-base-uncased',
                 num_labels: int = 2,
                 hidden_dim: int = 768,
                 dropout_prob: float = 0.1):
        super().__init__()
        # Базовая модель BERT без головы для маскированного языка
        self.bert = BertModel.from_pretrained(pretrained_model_name)

        # Кастомная голова
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_prob),
            nn.Linear(self.bert.config.hidden_size, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self,
                input_ids: torch.LongTensor,
                attention_mask: torch.Tensor = None,
                token_type_ids: torch.Tensor = None,
                labels: torch.LongTensor = None):
        # Получаем выходы из берта
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )
        # Выбираем pooled_output
        pooled_output = outputs.pooler_output

        # Передаём через свою голову
        logits = self.classifier(pooled_output)

        # Если есть метки - считаем loss
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
            return {
                'loss': loss,
                'logits': logits
            }
        return {'logits': logits}'''

In [ ]:
'''import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import BertTokenizer, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from dataclasses import dataclass

combined_df = abc
combined_df['sarcasm'] = combined_df['sarcasm'].astype(int)

train_df, test_df = train_test_split(
    combined_df,
    test_size=0.2,
    stratify=combined_df['sarcasm'],
    random_state=42
)

# Токенизация
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

@dataclass
class NERFeatures:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    token_type_ids: torch.Tensor
    labels: torch.Tensor

class SarcasmDataset(TorchDataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df['text'].tolist()
        self.labels = df['sarcasm'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        enc = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'token_type_ids': enc.get('token_type_ids', torch.zeros_like(enc['input_ids'])).squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Создаем модели
train_dataset = SarcasmDataset(train_df, tokenizer)
test_dataset  = SarcasmDataset(test_df, tokenizer)

# Загрузка модели
model = CustomBertClassifier(
    pretrained_model_name='bert-base-uncased',
    num_labels=2,
    hidden_dim=768,
    dropout_prob=0.1
)

# Метрики

def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='macro')
    }

# Параметры
training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2,

    # частота в шагах
    logging_steps=50,
    eval_steps=500,
    save_steps=500,
    save_total_limit=1,

    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
)

# Трейнер
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()
results = trainer.evaluate()
print(results)
# Save the best model
trainer.save_model('./best_model')'''
